[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/89-self-consistency/self_consistency_workbook.ipynb)

# 89 · Self-Consistency — Sample N Paths, Take the Majority Vote

## Wang et al. 2022 — Majority Voting over Sampled Reasoning Chains

**⏱ ~90 min**

Self-consistency (Wang et al. 2022) improves LLM reasoning by sampling multiple independent reasoning chains at temperature > 0 and taking the majority vote — like asking N experts and picking the answer most agreed on. This workbook builds the full pipeline using LangGraph's Send() fan-out and Annotated list reducers.

---

## Workshop Roadmap

| Part | Topic | Cells |
|------|-------|-------|
| 1 | What is self-consistency and why majority vote beats greedy decode | 3–4 |
| 2 | The Send() fan-out pattern with Annotated[list, operator.add] reducer | 5–6 |
| 3 | Temperature sampling — why temp > 0 matters | 7–8 |
| 4 | Building the full graph: dispatch → reason (×N) → aggregate | 9–10 |
| 5 | Accuracy analysis: vote distribution with N=1,3,5 | 11–12 |
| 6 | Exercises | 13–15 |

---

## Prerequisites

- Python 3.10+
- OpenAI API key
- Familiarity with LangGraph basics

**Reference paper:** Wang et al. 2022 — Self-Consistency Improves Chain of Thought Reasoning in Language Models  
arxiv.org/abs/2203.11171


In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        'langchain', 'langchain-openai', 'langgraph', 'python-dotenv'])
    print('Colab install complete.')
else:
    print('Local — skipping install.')


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get('OPENAI_API_KEY', '')
key_ok = bool(key) and key.startswith('sk-')
print(f'OPENAI_API_KEY present and valid-looking: {key_ok}')
if not key_ok:
    raise RuntimeError('OPENAI_API_KEY is required for the live self-consistency cells.')


## Part 1 — What is self-consistency and why majority vote beats greedy decode

**Greedy decode** picks the highest-probability token at each step (temperature=0). It produces a single, deterministic reasoning path. If that path makes one wrong inference, the final answer is wrong — there is no recovery.

**Self-consistency** samples N independent chains at temperature > 0. Each chain explores a slightly different route through reasoning space. The most common final answer across chains is returned as the prediction.

Think of it like asking N experts independently, then tallying votes. Even if a few experts reason incorrectly, the majority tends to arrive at the right answer.

| Property | Greedy decode | Self-consistency |
|----------|--------------|------------------|
| Paths sampled | 1 | N (typically 5–40) |
| Temperature | 0 (deterministic) | > 0 (diverse) |
| Variation across runs | None | High (by design) |
| Robustness to one bad step | Brittle | Robust |
| API cost | 1× | N× |
| Best for | Speed, simple tasks | Reasoning, math, logic |

The key insight: correct reasoning paths outnumber incorrect ones when the model has reasonable capability on the task. Majority vote exploits this asymmetry.


In [ ]:
from collections import Counter

# Mock reasoning chains — in real SC these come from N LLM samples
mock_answers = ['42', '42', '41', '42', '43', '42', '41']

def majority_vote(answers: list[str]) -> tuple[str, dict]:
    counts = Counter(answers)
    winner = counts.most_common(1)[0][0]
    return winner, dict(counts)

winner, distribution = majority_vote(mock_answers)
print(f'Answers: {mock_answers}')
print(f'Distribution: {distribution}')
print(f'Majority vote winner: "{winner}"')
print(f'Confidence: {distribution[winner]}/{len(mock_answers)} = {distribution[winner]/len(mock_answers):.1%}')


## Part 2 — The Send() fan-out pattern with Annotated[list, operator.add] reducer

### Send() API

`Send('node_name', state_dict)` schedules one independent invocation of the named node with the given private state. Returning a list of Send objects from a conditional entry point spawns all of them in parallel.

```python
def dispatch(state: SCState) -> list[Send]:
    return [Send('reason', {'problem': state['problem'], 'path_id': i, ...})
            for i in range(N_PATHS)]
```

### Annotated[list, operator.add] reducer

By default, LangGraph overwrites state keys when a node returns an update. For fan-out, we want each parallel path to **append** its result, not overwrite the others. `Annotated[list[dict], operator.add]` tells LangGraph to concatenate lists:

- path_0 returns `{'paths': [result_0]}`
- path_4 returns `{'paths': [result_4]}`
- LangGraph applies `operator.add` → `paths == [result_0, result_1, result_2, result_3, result_4]`

### Graph topology

```
         ┌──────────────────────┐
         │      dispatch        │  (conditional entry point)
         │  returns N Send()s   │
         └──────────────────────┘
          /    |    |    |    \
         ↓    ↓    ↓    ↓    ↓
       r(0) r(1) r(2) r(3) r(4)   (reason node, N parallel copies)
          \    |    |    |    /
           ↓   ↓   ↓   ↓   ↓
         ┌──────────────────────┐
         │      aggregate       │  (majority vote)
         └──────────────────────┘
                    ↓
                   END
```


In [ ]:
import operator
from typing import Annotated
from typing_extensions import TypedDict

N_PATHS = 5  # number of independent reasoning chains to sample

class PathState(TypedDict):
    'Private state for each sampled reasoning path.'
    problem: str
    path_id: int
    reasoning: str   # chain-of-thought steps
    answer: str      # extracted final answer

class SCState(TypedDict):
    'Root state for the full self-consistency graph.'
    problem: str
    # Annotated + operator.add: each parallel path appends its result; no overwriting
    paths: Annotated[list[dict], operator.add]
    final_answer: str

print('PathState fields:', list(PathState.__annotations__.keys()))
print('SCState fields:', list(SCState.__annotations__.keys()))
print()
print('Key: Annotated[list[dict], operator.add]')
print('  path_0 returns {"paths": [r0]}')
print('  path_4 returns {"paths": [r4]}')
print('  LangGraph applies operator.add -> paths == [r0, r1, r2, r3, r4]')


## Part 3 — Temperature sampling: why temp > 0 matters

Temperature controls the randomness of the model's token sampling:

- **temp=0** → deterministic, always picks the highest-probability token. All N paths produce identical reasoning. Majority vote degenerates to greedy decode — no benefit.
- **temp=0.5** → moderate diversity. Intermediate steps vary; final answers often still converge. Good balance of diversity and coherence.
- **temp=1.0** → high diversity. Strong exploration, but risk of incoherent chains increases. Works well for problems with clear correct answers.

Wang et al. recommend **temp=0.7** as a good default for most reasoning tasks.

| Temperature | Path diversity | Risk of incoherence | SC benefit |
|-------------|---------------|---------------------|------------|
| 0.0 | None (all identical) | None | Zero |
| 0.5 | Moderate | Low | Moderate |
| 0.7 | High | Low–medium | High |
| 1.0 | Very high | Medium | High (but noisy) |

Key insight: intermediate reasoning steps differ across paths, but many paths still converge on the same correct final answer — that is the diversity we exploit.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# temperature=0.7 produces diverse reasoning chains — the whole point of self-consistency
sampler_llm = ChatOpenAI(model='gpt-5.4-nano', temperature=0.7)

REASON_PROMPT = (
    'Solve this step by step. Show your reasoning.\n'
    'At the end, write "Answer: X" where X is the final answer only.\n\n'
    'Problem: {problem}\n'
    'Path {path_id} of {n_paths}: approach it naturally, show all steps.'
)

demo_problem = (
    'If a train travels 120 miles in 2 hours, then slows to half speed '
    'for 1 more hour, how far has it traveled total?'
)

print(f'Problem: {demo_problem}')
print(f'Sampling 3 reasoning paths at temperature=0.7...\n')

for i in range(3):
    prompt = REASON_PROMPT.format(problem=demo_problem, path_id=i+1, n_paths=3)
    resp = sampler_llm.invoke([HumanMessage(content=prompt)])
    lines = resp.content.strip().split('\n')
    print(f'--- Path {i+1} ---')
    for line in lines:
        print(f'  {line}')
    print()


## Part 4 — Building the full graph: dispatch → reason (×N) → aggregate

### Graph topology (full detail)

```
START
  ↓
[conditional entry point: dispatch()]
  returns list[Send('reason', PathState)] — one per path
  ↓  ↓  ↓  ↓  ↓   (N parallel branches)
[reason node] × N
  each returns {'paths': [one_result]}
  operator.add merges all into SCState.paths
  ↓
[aggregate node]
  Counter over paths[*].answer → majority_vote
  returns {'final_answer': winner}
  ↓
END
```

### Node responsibilities

| Node | Input | Output | Notes |
|------|-------|--------|-------|
| dispatch | SCState | list[Send] | Fan-out only, no LLM call |
| reason | PathState | dict with 'paths' key | One LLM call, extracts answer |
| aggregate | SCState (paths populated) | dict with 'final_answer' | Counter + most_common |


In [ ]:
import re
from collections import Counter
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.types import Send

sampler_llm = ChatOpenAI(model='gpt-5.4-nano', temperature=0.7)  # diverse sampling

REASON_PROMPT = (
    'Solve this problem step by step. Show all reasoning.\n'
    'End your response with exactly: Answer: <answer>\n\n'
    'Problem: {problem}\n'
    '(Reasoning chain {path_id} of {n_paths})'
)

AGG_PROMPT = (
    'The following are {n} independent solutions to a problem.\n'
    'Take the majority vote: identify the most common final answer.\n'
    'Return ONLY the winning answer, nothing else.\n\n'
    'Problem: {problem}\n\n'
    'Solutions:\n{solutions}\n\n'
    'Most common answer:'
)

def dispatch(state: SCState) -> list[Send]:
    'Fan out N independent reasoning paths using Send().'
    # list[Send] from a conditional entry point = LangGraph parallel fan-out
    return [
        Send('reason', {'problem': state['problem'], 'path_id': i, 'reasoning': '', 'answer': ''})
        for i in range(N_PATHS)
    ]

def reason(state: PathState) -> dict:
    'Sample one reasoning chain and extract the final answer.'
    prompt = REASON_PROMPT.format(
        problem=state['problem'], path_id=state['path_id']+1, n_paths=N_PATHS)
    resp = sampler_llm.invoke([HumanMessage(content=prompt)])
    text = resp.content.strip()
    # Extract 'Answer: X' from the end of the chain
    match = re.search(r'Answer:\s*(.+?)$', text, re.IGNORECASE | re.MULTILINE)
    answer = match.group(1).strip() if match else text.split('\n')[-1].strip()
    print(f'  Path {state["path_id"]+1}: answer="{answer}"')
    return {'paths': [{'path_id': state['path_id'], 'reasoning': text, 'answer': answer}]}

def aggregate(state: SCState) -> dict:
    'Majority vote over all sampled answers.'
    answers = [p['answer'] for p in state['paths']]
    counts = Counter(answers)
    winner = counts.most_common(1)[0][0]
    print(f'  Vote: {dict(counts)} -> winner: "{winner}"')
    return {'final_answer': winner}

# Build graph
graph = StateGraph(SCState)
graph.add_node('reason', reason)
graph.add_node('aggregate', aggregate)
graph.set_conditional_entry_point(dispatch, {'reason': 'reason'})
graph.add_edge('reason', 'aggregate')
graph.add_edge('aggregate', END)
app = graph.compile()
print('Graph compiled: dispatch -> reason(xN) -> aggregate -> END')


## Part 5 — Accuracy analysis: vote distribution with N=1, 3, 5

How does the number of sampled paths affect reliability?

- **N=1**: Exactly one sample — equivalent to temperature-sampled greedy decode. No majority vote benefit. High variance.
- **N=3**: Starts to help. A 2-1 majority is weak but better than a coin flip. Useful for low-stakes tasks.
- **N=5**: Solid majority. A 3-2 split still gives a winner; 4-1 or 5-0 is highly confident. This is the recommended minimum.
- **N=10+**: Strong reliability for hard reasoning. Diminishing returns past ~20 for most tasks.

The cell below runs a reasoning problem with N=1, 3, 5 so you can observe how vote distributions evolve.


In [ ]:
def make_sc_app(n_paths: int):
    'Create a self-consistency pipeline with n_paths reasoning chains.'
    def _dispatch(state: SCState) -> list[Send]:
        return [
            Send('reason_n', {'problem': state['problem'], 'path_id': i, 'reasoning': '', 'answer': ''})
            for i in range(n_paths)
        ]
    def _reason_n(state: PathState) -> dict:
        prompt = REASON_PROMPT.format(
            problem=state['problem'], path_id=state['path_id']+1, n_paths=n_paths)
        resp = sampler_llm.invoke([HumanMessage(content=prompt)])
        text = resp.content.strip()
        match = re.search(r'Answer:\s*(.+?)$', text, re.IGNORECASE | re.MULTILINE)
        answer = match.group(1).strip() if match else text.split('\n')[-1].strip()
        return {'paths': [{'path_id': state['path_id'], 'reasoning': text, 'answer': answer}]}

    g = StateGraph(SCState)
    g.add_node('reason_n', _reason_n)
    g.add_node('aggregate', aggregate)
    g.set_conditional_entry_point(_dispatch, {'reason_n': 'reason_n'})
    g.add_edge('reason_n', 'aggregate')
    g.add_edge('aggregate', END)
    return g.compile()

test_problem = (
    'A box has 3 red balls and 5 blue balls. '
    'You draw 2 balls without replacement. '
    'What is the probability both are red?'
)
print(f'Problem: {test_problem}\n')

for n in [1, 3, 5]:
    print(f'N_PATHS={n}:')
    trial = make_sc_app(n)
    result = trial.invoke({'problem': test_problem, 'paths': [], 'final_answer': ''})
    answers = [p['answer'] for p in result['paths']]
    counts = Counter(answers)
    print(f'  Answers: {answers}')
    print(f'  Vote distribution: {dict(counts)}')
    print(f'  Winner: "{result["final_answer"]}"')
    print()


## Part 6 — Exercises

### Exercise (a): Increase N_PATHS to 7

Change `N_PATHS` to 7 and re-run the full pipeline on `test_problem`. Observe:
- Does the vote spread change?
- Does the majority become more decisive?
- How does cost (API calls) scale?

### Exercise (b): Try temperature=0 — what happens?

Change `sampler_llm` to `ChatOpenAI(model='gpt-5.4-nano', temperature=0)` and re-run with N=5.
- What do you notice about the vote distribution?
- Does majority vote still help?
- What does this tell you about the relationship between temperature and self-consistency?


In [ ]:
# Exercise (a): Change N_PATHS to 7 and run the graph
# TODO: set N_PATHS_EX = 7 and call make_sc_app(N_PATHS_EX)

N_PATHS_EX = 5  # TODO: change to 7

ex_problem = (
    'A farmer has 17 sheep. All but 9 run away. '
    'How many sheep does the farmer have left?'
)

print('Exercise (a): N_PATHS =', N_PATHS_EX)
ex_app_a = make_sc_app(N_PATHS_EX)
ex_result_a = ex_app_a.invoke({'problem': ex_problem, 'paths': [], 'final_answer': ''})
ex_answers_a = [p['answer'] for p in ex_result_a['paths']]
print('  Answers:', ex_answers_a)
print('  Distribution:', dict(Counter(ex_answers_a)))
print('  Winner:', ex_result_a['final_answer'])

print()

# Exercise (b): Try temperature=0 — observe vote distribution collapse
# TODO: change temperature below from 0.7 to 0.0 and observe

temp_to_test = 0.0

print(f'Exercise (b): temperature={temp_to_test}')
ex_llm_b = ChatOpenAI(model='gpt-5.4-nano', temperature=temp_to_test)

def _reason_b(state: PathState) -> dict:
    prompt = REASON_PROMPT.format(
        problem=state['problem'], path_id=state['path_id']+1, n_paths=5)
    resp = ex_llm_b.invoke([HumanMessage(content=prompt)])
    text = resp.content.strip()
    match = re.search(r'Answer:\s*(.+?)$', text, re.IGNORECASE | re.MULTILINE)
    answer = match.group(1).strip() if match else text.split('\n')[-1].strip()
    return {'paths': [{'path_id': state['path_id'], 'reasoning': text, 'answer': answer}]}

g_b = StateGraph(SCState)
g_b.add_node('reason_b', _reason_b)
g_b.add_node('aggregate', aggregate)
g_b.set_conditional_entry_point(
    lambda s: [Send('reason_b', {'problem': s['problem'], 'path_id': i, 'reasoning': '', 'answer': ''})
               for i in range(5)],
    {'reason_b': 'reason_b'}
)
g_b.add_edge('reason_b', 'aggregate')
g_b.add_edge('aggregate', END)
app_b = g_b.compile()

ex_result_b = app_b.invoke({'problem': ex_problem, 'paths': [], 'final_answer': ''})
ex_answers_b = [p['answer'] for p in ex_result_b['paths']]
print('  Answers:', ex_answers_b)
print('  Distribution:', dict(Counter(ex_answers_b)))
print('  Winner:', ex_result_b['final_answer'])
print()
print('Observation: at temp=0 all paths return identical answers -> distribution collapses to N/N.')


## Answer Key

### Exercise (a): N_PATHS = 7

With N_PATHS=7 you get more votes in total. Expected observations:

- The vote distribution typically spreads more (more unique answer variants may appear with more paths).
- The winning answer's margin usually **increases** — the majority becomes more decisive.
- A 5-2 or 6-1 split is much more confident than a 3-2 split at N=5.
- API cost scales linearly: 7 LLM calls instead of 5.
- For most problems, N=7 gives a meaningful confidence boost over N=5 at a modest extra cost.

### Exercise (b): temperature = 0.0

With temperature=0, the model is fully deterministic: every token at every step picks the single highest-probability option. This means:

- All N paths produce **identical** reasoning chains and identical final answers.
- The vote distribution collapses to **N/N** (e.g., 5/5 for one answer).
- The vote spread is **zero** — every vote goes to the same candidate.
- Majority vote degenerates to **greedy decode**: the result is exactly what you would get with N=1 at temp=0.
- There is no diversity to exploit, so self-consistency provides **no benefit** over a single greedy sample.

**Takeaway**: Temperature diversity is not a side-effect of self-consistency — it is the mechanism. Without it, the technique collapses.
